In [ ]:
from pathlib import Path
import pandas as pd

# Solo archivos de calificaciones; el encabezado está en la fila 7 del Excel
archivos = sorted(Path("../data/algebra").glob("calificaciones*.xlsx"))
dfs = {a.name: pd.read_excel(a, header=6) for a in archivos}

In [ ]:
for nombre, df in dfs.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    filas_nulas = df.isna().all(axis=1)
    sin_nombre = df[["apellido", "nombre"]].isna().any(axis=1) & ~filas_nulas
    sin_usuario = df["usuario"].isna() & ~filas_nulas
    print(nombre)
    print(f"  Filas totalmente nulas: {filas_nulas.sum()}")
    print(f"  Filas sin nombre/apellido: {sin_nombre.sum()}")
    print(f"  Filas sin usuario: {sin_usuario.sum()}")

    # Se eliminan filas y columnas totalmente nulas, y filas sin usuario, nombre o apellido
    dfs[nombre] = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])

In [ ]:
tipos_evaluacion = {10: "autoevaluacion", 20: "recuperatorio", 25: "parcial"}

for nombre, df in dfs.items():
    # La nota máxima es la misma para todo el archivo: se toma de las notas no nulas
    nota_max = df["nota"].dropna().str.split("/").str[1].mode()[0]
    print(f"{nombre}: notas nulas = {df['nota'].isna().sum()}")
    df["nota"] = df["nota"].fillna(f"0/{nota_max}")

    df[["nota_obtenida", "nota_max"]] = df["nota"].str.split("/", expand=True).astype(float)
    df["tipo_evaluacion"] = df["nota_max"].map(tipos_evaluacion)

In [ ]:
# Se guarda el archivo de origen para distinguir evaluaciones del mismo tipo (ej. autoevaluación 1 y 2)
df_calificaciones = pd.concat(dfs, names=["archivo"]).reset_index(level="archivo").reset_index(drop=True)

df_calificaciones.to_parquet("../data/algebra/calificaciones_algebra.parquet", index=False)
df_calificaciones

In [ ]:
def a_snake_case(columnas):
    # Minúsculas, sin tildes y con "_" en lugar de espacios u otros símbolos
    return (columnas.str.strip().str.lower()
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
            .str.replace(r"\W+", "_", regex=True))


def limpiar_reporte(df):
    df.columns = a_snake_case(df.columns)
    df = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])
    # Se descartan usuarios de staff y docentes
    return df[~df["usuario"].str.startswith(("STAFF", "DOC"))].reset_index(drop=True)

In [ ]:
archivo = next(Path("../data/algebra").glob("reporte_participacion*.xlsx"))
df_participacion = pd.read_excel(archivo, header=[5, 6])
# Encabezado de dos niveles (ej. "Encuestas" / "Cantidad"): se unen en un solo nombre
df_participacion.columns = pd.Index(
    [" ".join(n for n in col if not n.startswith("Unnamed")) for col in df_participacion.columns]
)
df_participacion = limpiar_reporte(df_participacion)

df_participacion.to_parquet("../data/algebra/participacion_algebra.parquet", index=False)
df_participacion

In [ ]:
archivo = next(Path("../data/algebra").glob("reporte_trazabilidad*.xlsx"))
df_trazabilidad = limpiar_reporte(pd.read_excel(archivo, header=5))

df_trazabilidad.to_parquet("../data/algebra/trazabilidad_algebra.parquet", index=False)
df_trazabilidad